In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler , OneHotEncoder
import numpy as np
import pickle



In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)

In [4]:
label_encoder_gender = LabelEncoder()
data['gender']=label_encoder_gender.fit_transform(data['Gender'])

In [5]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,gender
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0


In [6]:
one_hot_encoded_geo= OneHotEncoder()
geo_encoder = one_hot_encoded_geo.fit_transform(data[['Geography']])

In [7]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encoded_geo.get_feature_names_out(['Geography']))


In [9]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
data = pd.concat([data.drop(['Geography', 'Gender'], axis=1), geo_encoded_df], axis=1)


In [11]:
with open('label_encoder_gender.pkl','wb') as file:
   pickle.dump(label_encoder_gender,file)

In [12]:
with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoded_geo,file)


In [13]:
data.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,gender,Geography_France,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,101348.88,1,0,1.0,0.0,0.0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,0.0,0.0,1.0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,1.0,0.0,0.0
3,699,39,1,0.00,2,0,0,93826.63,0,0,1.0,0.0,0.0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,0.0,0.0,1.0


In [14]:
X= data.drop('Exited',axis=1)
y=data['Exited']

In [15]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
scaler=StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [18]:
import tensorflow as tf


In [19]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [20]:
model= Sequential([
    Dense(64,activation='relu',input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [21]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
#compile the model

opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [23]:
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [24]:
log_dir="logs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [25]:
tensorboard_callback

In [26]:
early_stopping_callback=EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [27]:
#training the model
history=model.fit(X_train, y_train, epochs=100,validation_data=(X_test, y_test),validation_split=0.2, callbacks=[early_stopping_callback, tensorboard_callback], batch_size=32)


Epoch 1/100


250/250 [==============================] - 1s 2ms/step - loss: 0.4047 - accuracy: 0.8292 - val_loss: 0.3761 - val_accuracy: 0.8515
Epoch 2/100
250/250 [==============================] - 0s 997us/step - loss: 0.3557 - accuracy: 0.8565 - val_loss: 0.3586 - val_accuracy: 0.8475
Epoch 3/100
250/250 [==============================] - 0s 951us/step - loss: 0.3489 - accuracy: 0.8586 - val_loss: 0.3441 - val_accuracy: 0.8630
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3484 - accuracy: 0.8595 - val_loss: 0.3452 - val_accuracy: 0.8620
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3425 - accuracy: 0.8597 - val_loss: 0.3354 - val_accuracy: 0.8650
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3389 - accuracy: 0.8586 - val_loss: 0.3421 - val_accuracy: 0.8605
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3365 - accuracy: 0.8591 - val_loss: 0.3494 - val_accuracy: 

In [28]:
model.save('model.h5')

d:\ANN- Churn modeling\myvenv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [29]:
#load tensorboard extension
%load_ext tensorboard

In [30]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6008 (pid 12184), started 2:44:56 ago. (Use '!kill 12184' to kill it.)